In [46]:
import os
import json
import random
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from skmultilearn.model_selection import iterative_train_test_split

from sklearn.metrics import (
    f1_score, hamming_loss, accuracy_score,
    classification_report
)
from sklearn.model_selection import RandomizedSearchCV
from funcs import *

In [48]:
def load_dataset(n_per_label=len(CHOSEN_LABELS)):
    """
    Returns:
        texts  : list of strings (prob_desc_description)
        labels : list of sets   (subset of LABELS)
    One entry per unique file. A file appearing in multiple labels
    is loaded once and carries all its matched labels.
    """
    file_to_labels = {}   # fname -> set of matched labels

    for label in CHOSEN_LABELS:
        for fname in fetch_files_per_label(label, n_per_label):
            file_to_labels.setdefault(fname, set()).add(label)

    texts, labels = [], []
    for fname, lbls in file_to_labels.items():
        path = os.path.join(DATA_DIR, fname)
        with open(path, encoding="utf-8") as f:
            rec = json.load(f)
        text = rec.get("prob_desc_description", "") or ""
        if text.strip():
            texts.append(text)
            labels.append(lbls)

    return texts, labels


# ── 2. binarize labels ────────────────────────────────────────────────────
def binarize(labels):
    mlb = MultiLabelBinarizer(classes=CHOSEN_LABELS)
    Y   = mlb.fit_transform(labels)
    return Y, mlb

# split
def split(texts, Y, test_size=0.2, val_size=0.1, seed=42):
    np.random.seed(seed)
    idx = np.arange(len(texts)).reshape(-1, 1)

    idx_tv, Y_tv, idx_test, Y_test = iterative_train_test_split(idx, Y, test_size=test_size)
    idx_train, Y_train, idx_val, Y_val = iterative_train_test_split(
        idx_tv, Y_tv, test_size=val_size / (1 - test_size)
    )

    def pick(ix): return [texts[i[0]] for i in ix]
    return (pick(idx_train), Y_train), (pick(idx_val), Y_val), (pick(idx_test), Y_test)

# ── 4. TF-IDF ─────────────────────────────────────────────────────────────
def build_tfidf(train_texts):
    vec = TfidfVectorizer(
        ngram_range=(1, 2),   # unigrams + bigrams
        max_features=30_000,
        sublinear_tf=True,    # apply log(1+tf) — helps with long texts
        strip_accents="unicode",
        min_df=2,             # ignore terms appearing in < 2 docs
    )
    X_train = vec.fit_transform(train_texts)
    return vec, X_train


# ── 5. model ──────────────────────────────────────────────────────────────
def build_model():
    return OneVsRestClassifier(
        LogisticRegression(
            max_iter=1000,
            C=1.0,                    # inverse regularization strength
            class_weight="balanced",  # reweights for imbalanced labels
            solver="lbfgs",
        ),
        n_jobs=-1   # fit all binary classifiers in parallel
    )


# ── 6. metrics ────────────────────────────────────────────────────────────
def evaluate(model, X, Y_true, split_name="test"):
    Y_pred = model.predict(X)

    micro_f1  = f1_score(Y_true, Y_pred, average="micro",    zero_division=0)
    macro_f1  = f1_score(Y_true, Y_pred, average="macro",    zero_division=0)
    hloss     = hamming_loss(Y_true, Y_pred)
    subset_acc = accuracy_score(Y_true, Y_pred)   # exact match

    print(f"\n── {split_name} metrics ───────────────────────────────────")
    print(f"  Micro  F1      : {micro_f1:.4f}  (global — dominated by frequent labels)")
    print(f"  Macro  F1      : {macro_f1:.4f}  (avg per label — penalises rare failures)")
    print(f"  Hamming loss   : {hloss:.4f}  (fraction of wrong label bits, lower=better)")
    print(f"  Subset accuracy: {subset_acc:.4f}  (exact full-label match, strictest)")

    print(f"\n── per-label F1 ({split_name}) ────────────────────────────")
    print(classification_report(Y_true, Y_pred, target_names=CHOSEN_LABELS, zero_division=0))

    return {"micro_f1": micro_f1, "macro_f1": macro_f1,
            "hamming": hloss, "subset_acc": subset_acc}


# ── 7. threshold tuning ───────────────────────────────────────────────────
def tune_thresholds(model, X_val, Y_val, steps=20):
    """
    For each label, find the threshold on predicted probabilities
    that maximises F1 on the validation set.
    Returns an array of per-label thresholds.
    """
    probs      = model.predict_proba(X_val)
    thresholds = np.linspace(0.1, 0.9, steps)
    best_t     = np.full(len(CHOSEN_LABELS), 0.5)

    for j in range(len(CHOSEN_LABELS)):
        best_f1 = 0
        for t in thresholds:
            preds = (probs[:, j] >= t).astype(int)
            f1    = f1_score(Y_val[:, j], preds, zero_division=0)
            if f1 > best_f1:
                best_f1  = f1
                best_t[j] = t
        print(f"  {CHOSEN_LABELS[j]:<18} best threshold = {best_t[j]:.2f}  (F1={best_f1:.3f})")

    return best_t


def predict_with_thresholds(model, X, thresholds):
    probs = model.predict_proba(X)
    return (probs >= thresholds).astype(int)



In [49]:
print("Loading dataset...")
texts, labels = load_dataset()
print(f"  {len(texts)} unique files loaded")

Y, mlb = binarize(labels)
(train_texts, Y_train), (val_texts, Y_val), (test_texts, Y_test) = split(texts, Y)
print(f"  train={len(train_texts)}  val={len(val_texts)}  test={len(test_texts)}")

vec, X_train = build_tfidf(train_texts)
X_val        = vec.transform(val_texts)
X_test       = vec.transform(test_texts)

Loading dataset...
  2678 unique files loaded
  train=1880  val=266  test=532


In [50]:
def print_label_stats(Y_train, Y_val, Y_test, label_names):
    print(f"\n{'Label':<25} {'Train':>8} {'Val':>8} {'Test':>8} {'Total':>8}")
    print("-" * 60)
    for k, label in enumerate(label_names):
        tr = Y_train[:, k].sum()
        va = Y_val[:,   k].sum()
        te = Y_test[:,  k].sum()
        total = tr + va + te
        print(f"{label:<25}"
              f"{tr/total*100:>7.1f}%"
              f"{va/total*100:>7.1f}%"
              f"{te/total*100:>7.1f}%"
              f"{int(total):>8}")
    print("-" * 60)
    
print_label_stats(Y_train, Y_val, Y_test, CHOSEN_LABELS)


Label                        Train      Val     Test    Total
------------------------------------------------------------
math                        70.0%   10.0%   20.0%    1408
graphs                      70.1%   10.0%   19.9%     542
strings                     70.1%   10.0%   19.9%     422
number theory               70.0%   10.0%   20.0%     350
trees                       70.1%    9.9%   20.1%     324
geometry                    69.9%   10.2%   19.9%     166
games                       69.5%   10.5%   20.0%     105
probabilities               70.7%    9.8%   19.6%      92
------------------------------------------------------------


In [55]:
print("Training OneVsRest LogisticRegression...")
model = build_model()
model.fit(X_train, Y_train)

# evaluate with default threshold (0.5)
evaluate(model, X_val,  Y_val,  split_name="validation")
evaluate(model, X_test, Y_test, split_name="test")

# tune thresholds on val, re-evaluate on test
print("\nTuning per-label thresholds on validation set...")
thresholds = tune_thresholds(model, X_val, Y_val)

print("\n── test metrics after threshold tuning ──────────────────────")
Y_pred_tuned = predict_with_thresholds(model, X_test, thresholds)
print(f"  Micro  F1      : {f1_score(Y_test, Y_pred_tuned, average='micro',  zero_division=0):.4f}")
print(f"  Macro  F1      : {f1_score(Y_test, Y_pred_tuned, average='macro',  zero_division=0):.4f}")
print(f"  Hamming Score   : {1-hamming_loss(Y_test, Y_pred_tuned):.4f}")

print(classification_report(Y_pred_tuned, Y_test, target_names=CHOSEN_LABELS, zero_division=0))

model_logreg = model

Training OneVsRest LogisticRegression...

── validation metrics ───────────────────────────────────
  Micro  F1      : 0.7307  (global — dominated by frequent labels)
  Macro  F1      : 0.6525  (avg per label — penalises rare failures)
  Hamming loss   : 0.0869  (fraction of wrong label bits, lower=better)
  Subset accuracy: 0.5000  (exact full-label match, strictest)

── per-label F1 (validation) ────────────────────────────
               precision    recall  f1-score   support

         math       0.82      0.83      0.83       141
       graphs       0.55      0.57      0.56        54
      strings       0.85      0.79      0.81        42
number theory       0.55      0.63      0.59        35
        trees       0.65      0.81      0.72        32
     geometry       0.86      0.71      0.77        17
        games       0.69      0.82      0.75        11
probabilities       0.50      0.11      0.18         9

    micro avg       0.73      0.74      0.73       341
    macro avg     

### Random-Forest & GridSearch :

In [52]:
model = OneVsRestClassifier(
        RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced",   # handles label imbalance
            n_jobs=-1,                 # use all CPU cores
            random_state=42
        )
    )


param_grids = {
        "estimator__n_estimators":   [100, 300, 500],
        "estimator__max_depth":      [10, 20],
        "estimator__min_samples_leaf": [1, 2, 5],
        "estimator__max_features":   ["sqrt", "log2"]
    }




In [53]:
search = RandomizedSearchCV(
    estimator           = model,
    param_distributions = param_grids,  
    n_iter              = 20,
    cv                  = 5,
    scoring             = "f1_macro",
    n_jobs              = -1,
    random_state        = 42,
    verbose             = 1
)
search.fit(X_train, Y_train)

print(f"Best params : {search.best_params_}")
print(f"Best val F1 : {search.best_score_:.4f}")

# replace model with the best estimator found
model = search.best_estimator_


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best params : {'estimator__n_estimators': 500, 'estimator__min_samples_leaf': 5, 'estimator__max_features': 'sqrt', 'estimator__max_depth': 10}
Best val F1 : 0.5279


In [54]:
model.fit(X_train, Y_train)

# evaluate with default threshold (0.5)
# evaluate(model, X_val,  Y_val,  split_name="validation")
evaluate(model, X_test, Y_test, split_name="test")

# tune thresholds on val, re-evaluate on test
print("\nTuning per-label thresholds on validation set...")
thresholds = tune_thresholds(model, X_val, Y_val)

print("\n── test metrics after threshold tuning ──────────────────────")
Y_pred_tuned = predict_with_thresholds(model, X_test, thresholds)

print(f"  Micro  F1      : {f1_score(Y_test, Y_pred_tuned, average='micro',  zero_division=0):.4f}")
print(f"  Macro  F1      : {f1_score(Y_test, Y_pred_tuned, average='macro',  zero_division=0):.4f}")
print(f"  Hamming Score   : {1-hamming_loss(Y_test, Y_pred_tuned):.4f}")
print(classification_report(Y_pred_tuned, Y_test, target_names=CHOSEN_LABELS, zero_division=0))


── test metrics ───────────────────────────────────
  Micro  F1      : 0.7095  (global — dominated by frequent labels)
  Macro  F1      : 0.6624  (avg per label — penalises rare failures)
  Hamming loss   : 0.0987  (fraction of wrong label bits, lower=better)
  Subset accuracy: 0.4511  (exact full-label match, strictest)

── per-label F1 (test) ────────────────────────────
               precision    recall  f1-score   support

         math       0.74      0.85      0.79       282
       graphs       0.59      0.66      0.62       108
      strings       0.85      0.82      0.84        84
number theory       0.40      0.59      0.47        70
        trees       0.58      0.74      0.65        65
     geometry       0.81      0.76      0.78        33
        games       0.84      0.76      0.80        21
probabilities       0.80      0.22      0.35        18

    micro avg       0.67      0.75      0.71       681
    macro avg       0.70      0.67      0.66       681
 weighted avg   

## Conclusion : 
> best model so far : Logistic regression over TF-IDF representation

> to store and will be used in the CLI.

In [56]:
import joblib
import json
import numpy as np

# # best model from search
best_model = model_logreg
FOLDER = "best_model_All/"

# # save model
joblib.dump(best_model, FOLDER+"model.pkl")

# # save thresholds (assuming thresholds is a dict {label_name: float})
# # thresholds is already a dict {label_name: float} from tune_thresholds
with open(FOLDER+"thresholds.json", "w") as f:
    json.dump({k: float(v) for k, v in zip(CHOSEN_LABELS,thresholds)}, f, indent=2)

# saving the vectorier to ensure model sees same feature space.
joblib.dump(vec, FOLDER+"vectorizer.pkl")


['best_model_All/vectorizer.pkl']